<a href="https://colab.research.google.com/github/emily-escudero/Analitica-Educacion-rural/blob/main/Entrega2EducacionRural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests

# Definir la API URL
url = "https://www.datos.gov.co/api/v3/views/ji8i-4anb/query.json"

# solicitud a la API
respuesta_api = requests.get(url)
respuesta_api.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
datos = respuesta_api.json()


df = pd.DataFrame(datos)
display(df.head())

,:id,:version,:created_at,:updated_at,ano,c_digo_departamento,departamento,poblacion_5_16,tasa_matriculacion_5_16,cobertura_neta,...,reprobacion,reprobacion_transicion,reprobacion_primaria,reprobacion_secundaria,reprobacion_media,repitencia,repitencia_transicion,repitencia_primaria,repitencia_secundaria,repitencia_media
0,row-xccu-54yf.9ie2,rv-69kt.ykig.crwc,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,5,Antioquia,1288473,94.01,93.85,...,2.06,0.07,94.56,2.54,2.96,4.25,0.07,4.56,5.27,1.68
1,row-755p~4uyd-2k3w,rv-ar5t~pmwz_z3hy,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,8,Atlántico,523935,99.32,99.05,...,0.54,0.12,96.49,0.67,0.75,1.82,0.12,1.77,2.18,0.88
2,row-rb29.mx9s.a7ax,rv-atze~ze7f~jiue,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,11,"Bogotá, D.C.",1479334,90.7,90.29,...,0,0,94.69,0,0,3.23,0,2.3,5.11,2.57
3,row-gavf~qv4q~dhdy,rv-9idw~dvm2~k7me,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,13,Bolívar,496676,91.57,91.4,...,2.1,0.46,95.48,2.75,3.67,4.43,0.46,4.44,5.37,2.28
4,row-zz5g~58au~qymy,rv-68rp.uscp_a52j,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,15,Boyacá,300501,86.16,86.11,...,2.73,0.17,96.1,4.31,3.26,2.62,0.17,1.9,4.19,1.55


In [ ]:
# Definir columnas
columnas_desercion = [
    'departamento',
    'ano',
    'desercion',
    'desercion_transicion',
    'desercion_primaria',
    'desercion_secundaria',
    'desercion_media'
]

# Seleccionamos las columnas necesarias
datos_desercion = df[df.columns.intersection(columnas_desercion)].copy()

# Convertir la columna año a formato numerico
datos_desercion['ano'] = pd.to_numeric(datos_desercion['ano'], errors='coerce')

# Filtrar la informacion entre el 2021 y 2024
df_deserciones_2021_2024 = datos_desercion[
    (datos_desercion['ano'] >= 2021) & (datos_desercion['ano'] <= 2024)
].copy()

# Display the first few rows of the filtered DataFrame
display(df_deserciones_2021_2024.head())

,ano,departamento,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media
330,2021,Antioquia,4.81,3.59,4.27,5.90,4.13
331,2021,Atlántico,1.67,2.15,1.75,1.68,1.03
332,2021,"Bogotá, D,C,",1.29,1.07,1.08,1.35,1.88
333,2021,Bolívar,3.69,4.13,3.33,4.27,3.13
334,2021,Boyacá,2.97,2.75,2.13,3.63,3.69


In [ ]:
import pandas as pd
import requests

# URL del archivo excel en github
excel_url = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/anex-pobrezadepartamental.xlsx"

# Nombre del archivo
pobreza_departamental = "anex-pobrezadepartamental.xlsx"

# Descargar el archivo excel
print(f"Downloading {excel_url}...")
response = requests.get(excel_url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

with open(pobreza_departamental, 'wb') as f:
    f.write(response.content)

print(f"Archivo descargado {pobreza_departamental}")

Archivo descargado anex-pobrezadepartamental.xlsx


### Preparación de datos de pobreza monetaria

En este paso se extraen los datos de la hoja Pobreza Monetaria Act.Met. del archivo Excel. Los nombres de los departamentos se encuentran en la columna A, desde la fila 12 hasta la 35, mientras que los años están ubicados en la fila 11, desde las columnas B hasta F. Posteriormente, la información se organiza para facilitar su análisis y filtrado.

In [ ]:
# Load the 'Pobreza Monetaria Act.Met.' sheet
# We need to read the specific range for departments and lista_años.
# We'll read the data from the relevant cells and then process it.

# Leer la fila de loa lista_años
tabla_años = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Monetaria Act.Met.',
    header=None, # No default header
    skiprows=10, # Skip to row 11 (0-indexed)
    nrows=1,     # Read only one row for lista_años
    usecols='B:F' # Use columns B to F for lista_años
)

# Extraer los lista_años a la hoja de escel
lista_años = [int(col) for col in tabla_años.iloc[0].values]

# Leer los datos de pobreza monetaria por departamento
monetaria_df = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Monetaria Act.Met.',
    header=None, # No default header
    skiprows=11, # Skip to row 12 (0-indexed)
    nrows=24,    # From row 12 to 35 (35-12+1 = 24 rows)
    usecols='A:F' # Use columns A to F
)

# Asignar nombres a las columnas
monetaria_df.columns = ['departamento'] + lista_años

# convertir la tabla de formato ancho a formato largo
df_monetaria_melted = monetaria_df.melt(
    id_vars=['departamento'],
    var_name='ano',
    value_name='pobreza_monetaria'
)

# Filtrar la lista de años del 2021 al 2024
df_pobreza_monetaria = df_monetaria_melted[
    (df_monetaria_melted['ano'] >= 2021) & (df_monetaria_melted['ano'] <= 2024)
].copy()

# Display the first few rows
print("Pobreza Monetaria (2021-2024):")
display(df_pobreza_monetaria.head())

Pobreza Monetaria (2021-2024):


,departamento,ano,pobreza_monetaria
0,Antioquia,2021,32.8
1,Atlántico,2021,42.1
2,Bogotá D.C.,2021,30.5
3,Bolívar,2021,54.0
4,Boyacá,2021,41.8


### Preparacion de datos de pobreaza extrema

A continuación, se procesan los datos correspondientes a la hoja Pobreza Extrema Act.Met.. En esta hoja, los nombres de los departamentos se encuentran desde la celda A15 hasta A38, mientras que los años están ubicados en la fila 14, desde las columnas B hasta F.

In [ ]:

# Leer la fila donde se encuntran los años
tabla_años_extrema = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Extrema Act.Met.',
    header=None,
    skiprows=13,
    nrows=1,
    usecols='B:F'
)

# Extraer años de tabla_años_extrema
lista_años_extrema = [int(col) for col in tabla_años_extrema.iloc[0].values]

# Leer los datos de pobreza extrema por departamento
datos_pobreza_extrema = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Extrema Act.Met.',
    header=None,
    skiprows=14,
    nrows=24,
    usecols='A:F'
)

# Asignar nombres a las columnas
datos_pobreza_extrema.columns = ['departamento'] + lista_años_extrema

# Convertir la tabla de formato ancho a formato largo
df_extrema_melted = datos_pobreza_extrema.melt(
    id_vars=['departamento'],
    var_name='ano',
    value_name='pobreza_extrema'
)

# Filtrar años del 2021 al 2024
df_pobreza_extrema = df_extrema_melted[
    (df_extrema_melted['ano'] >= 2021) & (df_extrema_melted['ano'] <= 2024)
].copy()

# Mostrar primeras filas
print("Pobreza Extrema (2021-2024):")
display(df_pobreza_extrema.head())

Pobreza Extrema (2021-2024):


,departamento,ano,pobreza_extrema
0,Antioquia,2021,9.2
1,Atlántico,2021,12.3
2,Bogotá D.C.,2021,8.4
3,Bolívar,2021,18.8
4,Boyacá,2021,16.4


### Union base de datos

En este paso se integran los datos de pobreza monetaria y pobreza extrema en una sola base de datos, de manera que cada registro contenga la información correspondiente al mismo departamento y año.

In [ ]:
# Unir datos de pobreza monetaria a pobreza extrema
df_pobreza_completos = pd.merge(
    df_pobreza_monetaria,
    df_pobreza_extrema,
    on=['departamento', 'ano'],
    how='inner'
)

# Mostrar base de datos final
print("Final DataFrame with Pobreza Monetaria and Pobreza Extrema (2021-2024):")
display(df_pobreza_completos)

Final DataFrame with Pobreza Monetaria and Pobreza Extrema (2021-2024):


,departamento,ano,pobreza_monetaria,pobreza_extrema
0,Antioquia,2021,32.8,9.2
1,Atlántico,2021,42.1,12.3
2,Bogotá D.C.,2021,30.5,8.4
3,Bolívar,2021,54.0,18.8
4,Boyacá,2021,41.8,16.4
...,...,...,...,...
91,Risaralda,2024,23.8,5.2
92,Santander,2024,27.4,7.7
93,Sucre,2024,57.5,23.7
94,Tolima,2024,35.6,12.2


In [ ]:
import pandas as pd
import requests

# Url del archivo de educacion formal
educacion_formal = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/anex-educacionformal.xlsx"

# Nombre con el que guardo el archivo
ruta_educacion_formal = "anex-educacionformal.xlsx"

# Descargar archivo de excel
print(f"Downloading {educacion_formal}...")
respuesta_descarga = requests.get(educacion_formal)
respuesta_descarga.raise_for_status() # Raise an HTTPError for bad respuesta_descargas (4xx or 5xx)

with open(ruta_educacion_formal, 'wb') as f:
    f.write(respuesta_descarga.content)

print(f"Se dercargo el archivo {ruta_educacion_formal}")

Se dercargo el archivo anex-educacionformal.xlsx


### Preparación de Datos de Sedes Educativas

Ahora vamos a cargar los datos de la hoja `Sedes_nivel educativo_zona`. Según tus indicaciones:
*   Los departamentos están en la columna `A` desde la fila `19` hasta la `51`.
*   El número de sedes urbanas va desde la columna `B` hasta la `F` en la fila `17`.
*   El número de sedes rurales va desde la columna `G` hasta la `K` en la fila `17`.

In [ ]:

# 1. leer los nombres de las categorias de educacion
encabezados_educacion = pd.read_excel(
    ruta_educacion_formal,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=16,
    nrows=1,
    usecols='B:K'
)
nombres_educacion = encabezados_educacion.iloc[0].tolist()

# crear nombres para las columnas  urbanas y rurales
columnas_urbanas = [f'U_{name}' for name in nombres_educacion[:5]] # e.g., U_Preescolar
columnaas_rurales = [f'R_{name}' for name in nombres_educacion[5:]] # e.g., R_Preescolar
columnas_numericas = columnas_urbanas + columnaas_rurales


# 2. Leer los nombres de los departamentos
datos_departamentos = pd.read_excel(
    ruta_educacion_formal,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=18,
    nrows=33,
    usecols='A'
)
lista_departamentos = datos_departamentos.iloc[:, 0].tolist()

# 3. Leer los datos de sedes educativas urbanas y rurales
datos_sedes = pd.read_excel(
    ruta_educacion_formal,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=18,
    nrows=33,
    usecols='B:K'
)

# Asignar nombres a las columnas
datos_sedes.columns = columnas_numericas

#  DataFrame
df_instituciones_temp = pd.DataFrame(datos_sedes)
df_instituciones_temp.insert(0, 'departamento', lista_departamentos)

# Calcular el total de sedes educativas
df_instituciones_temp.iloc[:, 1:] = df_instituciones_temp.iloc[:, 1:].fillna(0)



df_instituciones = pd.DataFrame({
    'departamento': df_instituciones_temp['departamento'],
    'total_sedes_urbanas': df_instituciones_temp[columnas_urbanas].sum(axis=1),
    'total_sedes_rurales': df_instituciones_temp[columnaas_rurales].sum(axis=1)
})

df_instituciones['total_sedes'] = df_instituciones['total_sedes_urbanas'] + df_instituciones['total_sedes_rurales']

# Mostrar resultados
print("Total de Instituciones Urbanas y Rurales por Departamento:")
display(df_instituciones.head())
display(df_instituciones.tail())

Total de Instituciones Urbanas y Rurales por Departamento:


,departamento,total_sedes_urbanas,total_sedes_rurales,total_sedes
0,Amazonas,47,161,208
1,Antioquia,4126,9853,13979
2,Arauca,277,846,1123
3,"Archipiélago de San Andrés, Providencia y Sant...",46,35,81
4,Atlántico,3210,271,3481


,departamento,total_sedes_urbanas,total_sedes_rurales,total_sedes
28,Sucre,824,1573,2397
29,Tolima,1501,3488,4989
30,Valle del Cauca,4446,3065,7511
31,Vaupés,25,224,249
32,Vichada,48,462,510


### Análisis de Datos de Vivienda de TerriData

Ahora, trabajaré con el archivo `TerriData_vivienda.xlsx` para extraer y analizar los indicadores rurales solicitados.

In [ ]:
import io, requests, openpyxl, pandas as pd

url = "https://raw.githubusercontent.com/emily-escudero/Analitica-Educacion-rural/main/TerriData_vivienda.xlsx"
indicadores_rurales = [
    "Cobertura de energía eléctrica rural",
    "Cobertura de acueducto rural",
    "Déficit habitacional cualitativo en centros poblados y rural disperso",
    "Déficit habitacional cuantitativo centros poblados y rural disperso",
    "Déficit habitacional en centros poblados y rural disperso",
]

contenido = requests.get(url, timeout=120).content
ws = openpyxl.load_workbook(io.BytesIO(contenido), data_only=True, read_only=True)["Hoja01"]

registros = []
for cod_dep, dep, cod_ent, ent, dim, subcat, indicador, dato, _, anio, *_ in ws.iter_rows(min_row=2, values_only=True):
    if cod_dep is None or dep == "Colombia" or anio not in (2021, 2022, 2023, 2024):
        continue
    cod_dep = str(cod_dep).zfill(2)
    es_departamento = str(cod_ent) == cod_dep + "000" or (cod_dep == "11" and str(cod_ent) == "11001")
    if es_departamento and indicador in indicadores_rurales:
        valor = float(str(dato).replace(".", "").replace(",", ".")) if dato else None
        registros.append((dep, anio, indicador, valor))

df_vivienda_rural = pd.DataFrame(registros, columns=["departamento", "ano", "indicador", "valor"]).pivot(
    index=["departamento", "ano"], columns="indicador", values="valor"
).reset_index()

# Nota: San Andrés y Providencia no tiene dato en los indicadores de déficit habitacional rural
# (no tiene zona "centros poblados y rural disperso" catalogada). Es la única ausencia real.

df_vivienda_rural

indicador,departamento,ano,Cobertura de energía eléctrica rural,Déficit habitacional cualitativo en centros poblados y rural disperso,Déficit habitacional cuantitativo centros poblados y rural disperso,Déficit habitacional en centros poblados y rural disperso
0,Amazonas,2021,26.00,13.98,84.71,98.69
1,Amazonas,2022,26.75,5.60,93.68,99.28
2,Amazonas,2023,41.62,9.50,89.60,99.10
3,Amazonas,2024,27.11,2.70,94.10,96.80
4,Antioquia,2021,98.18,48.00,17.02,65.02
...,...,...,...,...,...,...
127,Vaupés,2024,49.21,2.20,97.80,100.00
128,Vichada,2021,10.83,38.98,54.13,93.11
129,Vichada,2022,13.36,32.94,52.83,85.77
130,Vichada,2023,12.15,21.40,66.40,87.80
